SHAP plots for SMOTE[nc] models

# Import libraries

In [1]:
import warnings
from numba.core.errors import NumbaDeprecationWarning
warnings.simplefilter(action='ignore', category=NumbaDeprecationWarning)

In [2]:
import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from itertools import combinations

import sys
from pathlib import Path
sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from modelling3_utils import (
    MLPipeline, StatsModelsEstimator)

from tqdm import tqdm
from sklearn.inspection import permutation_importance
from pathlib import Path
from statsmodels.api import Logit
import matplotlib.cm as cm
from IPython.display import clear_output

In [3]:
RANDOM_STATE = 42

# SHAP plots

In [4]:
path_shap_plots = Path('../results/shap_comparison')
path_shap_plots_splashing = path_shap_plots / 'splashing_smote_selection'
os.makedirs(path_shap_plots_splashing, exist_ok=True)
path_shap_plots_no_fragmentation = path_shap_plots / 'no_fragmentation_smote_selection'
os.makedirs(path_shap_plots_no_fragmentation, exist_ok=True)

In [5]:
smote_postfixes = [
    '_opt-model',
    '_smote_opt-model',
    '_smote_opt-smote_opt-model',
    '_smotenc_opt-model',
    '_smotenc_opt-smote_opt-model']

In [6]:
df_metrics = pd.read_excel(r'..\results\metrics_modelling4.xlsx')

In [7]:
split_types = ['test', 'train']
metrics = []
for split_type in split_types:
    metrics.extend(
        [
            f'holdout_{split_type}_f1_macro',
            f'holdout_{split_type}_roc_auc',
            f'holdout_{split_type}_accuracy_balanced',
            f'cv_{split_type}_f1_macro_median',
            f'cv_{split_type}_roc_auc_median',
            f'cv_{split_type}_accuracy_balanced_median',
        ]
    )

## Splashing

In [25]:
target = 'splashing'
df_splashing = df_metrics[df_metrics['target']==target]
# df_current_postfix = df_splashing[df_splashing['model'].apply(
#     lambda x: x.split(target)[-1]==smote_postfixes[0])]
# df_current_postfix = df_current_postfix.sort_values(
#     by=metrics[:6], ascending=False)
# df_current_postfix.head(3)

In [26]:
fnames_models = []
for postfix in smote_postfixes:
    df_current_postfix = df_splashing[df_splashing['model'].apply(
        lambda x: x.split(target)[-1]==postfix)]
    assert df_current_postfix.shape[0]>1
    df_current_postfix = df_current_postfix.sort_values(
        by=metrics[:6], ascending=False)
    fnames_models.append(df_current_postfix.iloc[0]["model"])
fnames_models

['CatBoostClassifier_splashing_opt-model',
 'CatBoostClassifier_splashing_smote_opt-model',
 'RandomForestClassifier_splashing_smote_opt-smote_opt-model',
 'LGBMClassifier_splashing_smotenc_opt-model',
 'AdaBoostClassifier_splashing_smotenc_opt-smote_opt-model']

In [ ]:
df_m_spl = df_splashing[df_splashing['model'].isin(fnames_models)]
df_m_spl[['dataset', 'target', 'model'] + [x for x in df_m_spl.columns if 'holdout' in x]].to_excel('../results/metrics_smote_best_models_splashing.xlsx', index=False)

In [ ]:
for fname in tqdm(fnames_models):
    index = 3
    if 'smote' in fname.lower():
        index = 4
    pipeline = joblib.load(f'../results/models_modelling4/{fname}')
    boost_model = pipeline.steps[index][1]
    ml_pipe = MLPipeline(
        target=target,
        estimator=boost_model,
        features_to_drop = (),
        model_postfix='test',)
    clear_output()
    X = ml_pipe.full_df.drop(columns=[target])
    X_transf = pipeline[:-2].transform(X)
    y = ml_pipe.full_df[target]
    if 'AdaBoost' in fname:
        explainer = shap.Explainer(boost_model.predict, X_transf)
    else:
        explainer = shap.Explainer(boost_model)
    shap_values = explainer(X_transf)
    if hasattr(X_transf, 'columns'):
        fnames = X_transf.columns
    elif hasattr(boost_model, 'feature_names_'): 
        fnames = boost_model.feature_names_
    elif hasattr(boost_model, 'feature_name_'): 
        fnames = boost_model.feature_name_
    elif hasattr(boost_model, 'feature_names_in_'):
        fnames = boost_model.feature_names_in_
    if len(shap_values.shape)==3:
        shap_values = shap_values[:, :, 0]
    shap.summary_plot(shap_values, X_transf, show=False, feature_names=fnames)
    short_name = fname.split('/')[-1]
    plt.title(short_name, fontsize=14)
    plt.savefig(path_shap_plots_splashing / f'{short_name}.jpg')
    plt.close()

ExactExplainer explainer: 373it [00:15,  8.52it/s]                         
100%|██████████| 5/5 [00:17<00:00,  3.44s/it]


## No fragmentation

In [21]:
target = 'no_fragmentation'
df_no_fragmentation = df_metrics[df_metrics['target']==target]
# df_current_postfix = df_no_fragmentation[df_no_fragmentation['model'].apply(
#     lambda x: x.split(target)[-1]==smote_postfixes[0])]
# df_current_postfix = df_current_postfix.sort_values(
#     by=metrics[:6], ascending=False)
# df_current_postfix.head(3)

In [22]:
fnames_models = []
for postfix in smote_postfixes:
    df_current_postfix = df_no_fragmentation[df_no_fragmentation['model'].apply(
        lambda x: x.split(target)[-1]==postfix)]
    assert df_current_postfix.shape[0]>1
    df_current_postfix = df_current_postfix.sort_values(
        by=metrics[:6], ascending=False)
    fnames_models.append(df_current_postfix.iloc[0]["model"])
fnames_models

['LGBMClassifier_no_fragmentation_opt-model',
 'RandomForestClassifier_no_fragmentation_smote_opt-model',
 'AdaBoostClassifier_no_fragmentation_smote_opt-smote_opt-model',
 'CatBoostClassifier_no_fragmentation_smotenc_opt-model',
 'RandomForestClassifier_no_fragmentation_smotenc_opt-smote_opt-model']

In [24]:
df_m_nofr = df_no_fragmentation[df_no_fragmentation['model'].isin(fnames_models)]
df_m_nofr[['dataset', 'target', 'model'] + [x for x in df_m_nofr.columns if 'holdout' in x]].to_excel('../results/metrics_smote_best_models_nofragmentation.xlsx', index=False)

In [68]:
for fname in tqdm(fnames_models):
    index = 3
    if 'smote' in fname.lower():
        index = 4
    pipeline = joblib.load(f'../results/models_modelling4/{fname}')
    boost_model = pipeline.steps[index][1]
    ml_pipe = MLPipeline(
        target=target,
        estimator=boost_model,
        features_to_drop = (),
        model_postfix='test',)
    clear_output()
    X = ml_pipe.full_df.drop(columns=[target])
    X_transf = pipeline[:-2].transform(X)
    y = ml_pipe.full_df[target]
    if 'AdaBoost' in fname:
        explainer = shap.Explainer(boost_model.predict, X_transf)
    else:
        explainer = shap.Explainer(boost_model)
    shap_values = explainer(X_transf)
    if hasattr(X_transf, 'columns'):
        fnames = X_transf.columns
    elif hasattr(boost_model, 'feature_names_'): 
        fnames = boost_model.feature_names_
    elif hasattr(boost_model, 'feature_name_'): 
        fnames = boost_model.feature_name_
    elif hasattr(boost_model, 'feature_names_in_'):
        fnames = boost_model.feature_names_in_
    if len(shap_values.shape)==3:
        shap_values = shap_values[:, :, 0]
    shap.summary_plot(shap_values, X_transf, show=False, feature_names=fnames)
    short_name = fname.split('/')[-1]
    plt.title(short_name, fontsize=14)
    plt.savefig(path_shap_plots_no_fragmentation / f'{short_name}.jpg')
    plt.close()

100%|██████████| 5/5 [04:20<00:00, 52.05s/it]
